# Prediction framework: will this incident reach 10+ minutes?

An incident has just been declared: how likely is it to reach a 10+ minute customer-service delay? The code is in [`models.py`](../src/ds_challenge/models.py), [`splits.py`](../src/ds_challenge/splits.py) and [`evaluation.py`](../src/ds_challenge/evaluation.py); this notebook runs it and reads the results.

- **Population:** train incidents on a single named line, 2019-2024. Multi-line rows, workshop and yard rows ("Non affecté") and the incomplete 2025 are left out (see the README, Data).
- **Target:** 10+ min, from the official delay bin. "Major" is not defined in the brief, so 10+ is an assumption.
- **Features, all known at declaration:** symptom, line, location, hour, weekday, month. Cause, power cut, evacuation, emergency team, resume time and duration are set after the fact and excluded. Emergency brake, door and material damage are left out because their timing is unknown.
- **Models:** the 10+ rate of the symptom (baseline), the 10+ rate of the symptom x location cell, and a logistic regression on all six features. Default settings, nothing tuned.
- **Validation:** each test year (2022, 2023, 2024) is scored by models fit on the years before it.
- **Metrics:** recall and precision when the 10% highest-scored incidents are flagged, and PR-AUC. The 10% alert budget is an assumption about how many incidents Ops can escalate.

In [1]:
import pandas as pd

from ds_challenge.data import Inc, load_incidents, model_population
from ds_challenge.evaluation import backtest, recall_lead
from ds_challenge.models import MODELS, TARGET

df = model_population(load_incidents())
df.groupby(Inc.year)[TARGET].agg(incidents="size", share_10_plus="mean").round(3)

,incidents,share_10_plus
year,,
2019,5014,0.051
2020,3663,0.067
2021,3515,0.070
2022,4162,0.066
2023,4428,0.076
2024,4974,0.073


## Backtest

Percent. A random 10% of alerts would give a recall of 10 and a precision and PR-AUC equal to the year's 10+ share (6.6, 7.6, 7.3).

In [2]:
results, alerts = backtest(df)
(results * 100).round(1).unstack("split")

recall             precision             pr_auc        \
split                 2022  2023  2024      2022  2023  2024   2022  2023   
model                                                                       
logistic_regression   21.2  20.3  20.0      13.9  15.4  14.7   10.8  12.8   
symptom               14.1  17.3  15.5       9.2  13.1  11.4    8.4   9.8   
symptom_location      22.0  16.8  20.4      14.4  12.7  15.0   12.8  12.2   

                           
split                2024  
model                      
logistic_regression  13.6  
symptom               9.1  
symptom_location     13.2

In [3]:
# recall over the three test years together, and each lead with its 95% paired bootstrap interval
pooled = alerts.loc[alerts[TARGET], list(MODELS)].mean()
pairs = [
    ("symptom_location", "symptom"),
    ("logistic_regression", "symptom"),
    ("logistic_regression", "symptom_location"),
]
leads = pd.DataFrame(
    [recall_lead(alerts, a, b) for a, b in pairs],
    columns=["lead", "ci_low", "ci_high"],
    index=[f"{a} - {b}" for a, b in pairs],
)
display((pooled * 100).round(1).to_frame("pooled recall"))
(leads * 100).round(1)

,pooled recall
symptom,15.7
symptom_location,19.6
logistic_regression,20.5


,lead,ci_low,ci_high
symptom_location - symptom,3.9,1.3,6.6
logistic_regression - symptom,4.7,2.2,7.1
logistic_regression - symptom_location,0.9,-2.3,3.8


## What the baseline uses

The 10+ rate by symptom, on 2019-2024.

In [4]:
df.groupby(Inc.symptom)[TARGET].agg(incidents="size", share_10_plus="mean").sort_values(
    "share_10_plus", ascending=False
).round(3)

,incidents,share_10_plus
symptom,,
Divers,91,0.132
"Feu, fumée, odeur, produit, etc...",1081,0.121
Équipements fixes,3469,0.112
Clientèle,11509,0.064
Matériel roulant,7020,0.062
not recorded,69,0.029
Exploitation stations,628,0.008
Exploitation trains,1889,0.007


## Reading

1. **The signal at declaration is weak.** Flagging 10% of incidents catches about 20% of the 10+ ones, against 15.7% with the symptom alone and 10% at random.
2. **The table and the regression tie:** 19.6% and 20.5%, each ahead of the symptom (+3.9, 1.3 to 6.6; +4.7, 2.2 to 7.1), 0.9 apart (-2.3 to 3.8).
3. **So it is a triage aid, not an escalation rule**.